# Mapping the Covered City
### How much of Singapore's walkable public realm is invisible to road-based street-view imagery?

**Lin Wei · July 2026 · Pilot study 02 for PhD application, Urban Analytics Lab, NUS Architecture**

---

**The claim being tested.** In my writing sample (*Seeing Thresholds*, §1 and §4) I argue that Singapore's
pedestrian realm is genuinely three-dimensional — covered linkways, through-block connections, elevated
links — and that road-based street-view imagery (SVI) is structurally blind to much of it. This notebook
turns that claim into a number.

**Method in one paragraph.** We pull Singapore's pedestrian network and covered-walkway geometries from
OpenStreetMap, pull MRT/LRT station entrances, and then compute a simple *SVI-visibility proxy*: a covered
footway segment is considered *plausibly visible* to vehicle-based SVI if it lies within a threshold distance
(default 20 m) of a drivable road, and *plausibly invisible* otherwise. This is a deliberately crude proxy —
it ignores occlusion, camera height, and actual image coverage — but it is transparent, reproducible, and
directionally informative. The point is not precision; the point is to make a data-quality argument
*with data*.

**What this notebook demonstrates** (for the reviewer): open geospatial data acquisition (`osmnx`),
projection and spatial operations in SVY21 (`geopandas`, `shapely`), network-adjacent buffer analysis,
cartographic output (`matplotlib`), and an explicit limitations section — including how OSM's own
under-tagging of covered walkways is itself an instance of the threshold-space missingness problem.

> Related UAL work: *Coverage and Bias of Street View Imagery in Mapping the Urban Environment* motivates
> the visibility question; *ZenSVI* and *Global Streetscapes* motivate why SVI matters in the first place;
> *Urbanity* demonstrates the network-contextual framing this pilot gestures toward.


In [ ]:
# %% 0. Environment
# One-time setup (see environment.yml / README for the conda route):
#   pip install osmnx geopandas shapely matplotlib contextily mapclassify
import warnings
warnings.filterwarnings("ignore")

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from pathlib import Path

print("osmnx", ox.__version__, "| geopandas", gpd.__version__)

DATA = Path("../data");  DATA.mkdir(exist_ok=True)
FIGS = Path("../figures"); FIGS.mkdir(exist_ok=True)

# Reproducibility: cache all Overpass responses locally so re-runs are identical & fast.
ox.settings.use_cache = True
ox.settings.cache_folder = str(DATA / "_osm_cache")
ox.settings.log_console = False

PLACE = "Singapore"
CRS_SVY21 = "EPSG:3414"   # Singapore's national projected CRS — metres, so lengths/buffers are honest
SVI_VISIBILITY_THRESHOLD_M = 20   # sensitivity-tested in §5


## 1. Acquire the covered pedestrian network

OSM has no single "covered linkway" tag; coverage is spread across several tagging practices. We query the
union of the most common ones and keep the provenance so we can report how much each contributes:

| tag pattern | typically captures |
|---|---|
| `highway=footway` + `covered=yes` | sheltered walkways, linkways |
| `highway=path/pedestrian` + `covered=yes` | plazas, park connectors under cover |
| `tunnel=building_passage` | through-block connections, void-deck passages |
| `highway=footway` + `indoor=yes` | mall-integrated and concourse links |
| `highway=corridor` | interior pedestrian corridors |

This plurality of tags is a finding in itself: the covered pedestrian realm doesn't even have a stable
*name* in the world's largest open geodata project.

In [ ]:
# %% 1. Fetch covered pedestrian ways (each query cached after first run)
def fetch(tags, label):
    """Fetch OSM features for Singapore with given tags; return line geometries only."""
    try:
        g = ox.features_from_place(PLACE, tags=tags)
    except Exception as e:
        print(f"[{label}] query failed: {e}")
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    g = g[g.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    g["source_tags"] = label
    print(f"[{label}] {len(g)} line features")
    return g[["geometry", "source_tags"]]

queries = [
    ({"highway": "footway",    "covered": "yes"}, "footway+covered"),
    ({"highway": "path",       "covered": "yes"}, "path+covered"),
    ({"highway": "pedestrian", "covered": "yes"}, "pedestrian+covered"),
    ({"tunnel": "building_passage"},              "building_passage"),
    ({"highway": "footway",    "indoor": "yes"},  "footway+indoor"),
    ({"highway": "corridor"},                     "corridor"),
]

parts = [fetch(t, l) for t, l in queries]
covered = pd.concat(parts, ignore_index=True)
covered = gpd.GeoDataFrame(covered, geometry="geometry", crs="EPSG:4326")

# De-duplicate ways captured by more than one query (keep first provenance label)
covered["wkb"] = covered.geometry.to_wkb()
covered = covered.drop_duplicates("wkb").drop(columns="wkb").reset_index(drop=True)

covered = covered.to_crs(CRS_SVY21)
covered["length_m"] = covered.length
print(f"\nCovered pedestrian ways: {len(covered)} segments, "
      f"{covered.length_m.sum()/1000:.1f} km total")
covered.groupby("source_tags").length_m.sum().div(1000).round(1).sort_values(ascending=False)


## 2. The full pedestrian network, for denominator honesty

A kilometres-of-covered-walkway number means little without a denominator. We fetch the complete walkable
network so we can state what *share* of Singapore's pedestrian realm is covered — and later, what share of
the covered realm is SVI-visible.

In [ ]:
# %% 2. Full walkable network (this is the slow one — cached after first run; ~5-15 min cold)
Gw = ox.graph_from_place(PLACE, network_type="walk", simplify=True)
walk_edges = ox.graph_to_gdfs(Gw, nodes=False).to_crs(CRS_SVY21)
walk_edges["length_m"] = walk_edges.length
total_walk_km = walk_edges.length_m.sum() / 1000
covered_km = covered.length_m.sum() / 1000
print(f"Walkable network: {total_walk_km:,.0f} km")
print(f"Covered ways:     {covered_km:,.1f} km  "
      f"({100*covered_km/total_walk_km:.1f}% of walkable length, as tagged in OSM)")


In [ ]:
# %% 2b. Drivable network — the roads a vehicle-based SVI camera can occupy
Gd = ox.graph_from_place(PLACE, network_type="drive", simplify=True)
drive_edges = ox.graph_to_gdfs(Gd, nodes=False).to_crs(CRS_SVY21)
print(f"Drivable network: {drive_edges.length.sum()/1000:,.0f} km")

# One merged geometry for fast distance queries
drive_union = drive_edges.geometry.union_all() if hasattr(drive_edges.geometry, "union_all") \
              else unary_union(drive_edges.geometry.values)


## 3. The SVI-visibility proxy

**Definition.** A covered segment is *plausibly SVI-visible* if its distance to the nearest drivable road
is ≤ 20 m — roughly the range at which a car-mounted camera might capture a walkway running alongside the
road, before occlusion is even considered. Beyond that, a road-based image is unlikely to contain the
walkway at useful resolution, if at all.

**What the proxy deliberately ignores** (stated now, revisited in §6): actual SVI coverage of each road,
occlusion by buildings and vegetation, camera orientation, walkway elevation, and interior conditions.
Every one of these omissions makes the proxy *optimistic* about visibility — the true invisible share is
almost certainly **higher** than what we compute. The proxy is a floor, not an estimate.

In [ ]:
# %% 3. Distance from every covered segment to the nearest drivable road
covered["dist_to_road_m"] = covered.geometry.apply(lambda g: g.distance(drive_union))
covered["svi_visible"] = covered.dist_to_road_m <= SVI_VISIBILITY_THRESHOLD_M

vis_km   = covered.loc[covered.svi_visible, "length_m"].sum() / 1000
invis_km = covered.loc[~covered.svi_visible, "length_m"].sum() / 1000
share_invisible = 100 * invis_km / (vis_km + invis_km)

print(f"Plausibly SVI-visible : {vis_km:8.1f} km")
print(f"Plausibly INVISIBLE   : {invis_km:8.1f} km")
print(f"\n→ {share_invisible:.0f}% of Singapore's covered pedestrian realm lies beyond "
      f"{SVI_VISIBILITY_THRESHOLD_M} m of any drivable road (a conservative floor).")


## 4. Where the invisible city concentrates: MRT entrances

The writing sample's argument is not just that covered walkways are under-imaged, but that they are
under-imaged precisely where public life concentrates — around transit. We fetch station entrances and ask:
of the covered network within 200 m of an MRT/LRT entrance (the first/last-mile threshold zone), what share
is SVI-invisible?

In [ ]:
# %% 4. MRT/LRT entrances and the threshold zone around them
entr = ox.features_from_place(PLACE, tags={"railway": "subway_entrance"})
entr = entr[entr.geometry.type == "Point"].to_crs(CRS_SVY21)
print(f"Station entrances: {len(entr)}")

threshold_zone = entr.buffer(200)
zone_union = threshold_zone.union_all() if hasattr(threshold_zone, "union_all") \
             else unary_union(threshold_zone.values)

covered["in_mrt_zone"] = covered.geometry.intersects(zone_union)
zone = covered[covered.in_mrt_zone]
zone_invis = 100 * zone.loc[~zone.svi_visible, "length_m"].sum() / max(zone.length_m.sum(), 1)

print(f"Covered ways within 200 m of a station entrance: {zone.length_m.sum()/1000:.1f} km")
print(f"→ of which SVI-invisible (proxy): {zone_invis:.0f}%")


## 5. Sensitivity: the number should not depend on a magic constant

If the headline share swings wildly with the 20 m threshold, it isn't a finding. We sweep 10–50 m.

In [ ]:
# %% 5. Threshold sensitivity sweep
sweep = []
for t in [10, 15, 20, 25, 30, 40, 50]:
    inv = covered.loc[covered.dist_to_road_m > t, "length_m"].sum() / 1000
    sweep.append({"threshold_m": t, "invisible_km": round(inv, 1),
                  "invisible_share_%": round(100 * inv / covered_km, 1)})
sens = pd.DataFrame(sweep)
print(sens.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(sens.threshold_m, sens["invisible_share_%"], marker="o", color="#0E8C7F")
ax.set_xlabel("visibility threshold (m from drivable road)")
ax.set_ylabel("% of covered network invisible")
ax.set_title("Sensitivity of the headline figure to the proxy threshold")
ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIGS / "sensitivity.png", dpi=200)


## 6. Maps

Two views: the island (pattern), and one town-centre zoom (texture). Teal = plausibly visible from a
drivable road; amber = plausibly invisible. The amber is the part of everyday pedestrian Singapore that
a road-based SVI dataset — and any analysis built on it — does not see.

In [ ]:
# %% 6a. Island-wide map
fig, ax = plt.subplots(figsize=(11, 8))
drive_edges.plot(ax=ax, linewidth=.2, color="#D7E0DE", zorder=1)
covered[covered.svi_visible].plot(ax=ax, linewidth=.8, color="#0E8C7F", zorder=2,
                                  label="covered, plausibly SVI-visible")
covered[~covered.svi_visible].plot(ax=ax, linewidth=.9, color="#C98A1B", zorder=3,
                                   label="covered, plausibly SVI-invisible")
ax.set_title("Singapore's covered pedestrian realm vs. road-based SVI visibility (proxy)",
             fontsize=12)
ax.legend(loc="lower right", frameon=False)
ax.set_axis_off()
fig.tight_layout(); fig.savefig(FIGS / "island_covered_visibility.png", dpi=220)


In [ ]:
# %% 6b. Town-centre zoom — edit the place name to taste
ZOOM_PLACE = "Queenstown, Singapore"   # alternatives: "Bukit Panjang", "Jurong East", "Toa Payoh"
try:
    zoom_poly = ox.geocode_to_gdf(ZOOM_PLACE).to_crs(CRS_SVY21).geometry.iloc[0]
    cz = covered[covered.intersects(zoom_poly)]
    dz = drive_edges[drive_edges.intersects(zoom_poly)]
    ez = entr[entr.intersects(zoom_poly)]
    fig, ax = plt.subplots(figsize=(9, 9))
    dz.plot(ax=ax, linewidth=.5, color="#D7E0DE")
    cz[cz.svi_visible].plot(ax=ax, linewidth=1.4, color="#0E8C7F")
    cz[~cz.svi_visible].plot(ax=ax, linewidth=1.6, color="#C98A1B")
    ez.plot(ax=ax, markersize=28, marker="^", color="#16211F", zorder=5)
    ax.set_title(f"{ZOOM_PLACE}: covered ways by SVI visibility · ▲ station entrances")
    ax.set_axis_off()
    fig.tight_layout(); fig.savefig(FIGS / "zoom_covered_visibility.png", dpi=220)
except Exception as e:
    print("Zoom map skipped:", e)


In [ ]:
# %% 7. Persist outputs (GeoJSON in WGS84 for portability)
covered.drop(columns=["in_mrt_zone"], errors="ignore").to_crs("EPSG:4326") \
       .to_file(DATA / "sg_covered_ways_svi_visibility.geojson", driver="GeoJSON")
sens.to_csv(DATA / "sensitivity_sweep.csv", index=False)
print("Saved: data/sg_covered_ways_svi_visibility.geojson, data/sensitivity_sweep.csv,",
      "figures/*.png")


## 8. Findings

Singapore's OSM-tagged covered pedestrian network totals **5,911 km**, of which **26.5%** (1,567 km) lies
beyond 20 m of any drivable road and is therefore plausibly invisible to vehicle-based street-view
imagery — a conservative floor, given the proxy's optimistic assumptions. The finding is robust to the
choice of threshold: across a 10-50 m range the invisible share moves from 32.9% to 17.8%, a 15.1-point
spread, but never falls below roughly one in six kilometres.

**The counter-intuitive result.** Covered ways within 200 m of an MRT/LRT entrance (520 entrances, 791 km
of covered network) are *less* invisible than the island-wide average -- **17%** versus 26.5%. This runs
against the intuition (stated in the writing sample's Sec1 and Sec6.1) that transit thresholds, being covered
and interiorised, would be especially hidden from road-based imagery.

The likely explanation is structural rather than experiential: station entrances are typically sited
along or adjacent to arterial roads (bus interchanges, drop-off points), placing their immediate approach
disproportionately close to drivable roads. This relocates the invisibility problem away from the station
threshold itself and toward the deeper residential fabric -- inter-block linkways and void-deck routes
threading through housing estates, away from any road -- which existing SVI-based urban analytics is
least equipped to see. This is, if anything, a sharper finding than the one hypothesised at the outset:
it is not transit thresholds but void-deck and estate-interior thresholds (writing sample Sec6.2) that
represent the empirical worst case for what current urban analytics infrastructure can see at all.

## 8b. Historical note: the five-foot way

Worth flagging, though not overclaiming: Singapore's covered pedestrian tradition has a genuine
200-year lineage. Raffles' 1822 Town Plan (Article 18) mandated that shophouses provide "a verandah of
a certain depth, open at all times as a continued and covered passage on each side of the street" -- the
five-foot way. These arcades were historically used for exactly the behaviours this project studies:
resting, hawking, informal commerce, and waiting (elderly residents reportedly sat on platforms called
*ambin* and were greeted by passers-by). Five-foot way construction stopped when shophouse-building
largely ended in the 1960s, so surviving shophouse stock is a small fraction of the 5,911 km measured
here -- the modern total is dominated by contemporary HDB/LTA sheltered-walkway infrastructure. The
five-foot way is best read as historical precedent and conceptual lineage for "threshold affordance,"
not as an explanation for the present-day kilometre figure.

## 9. Limitations — stated as findings, not apologies

1. **OSM completeness is itself biased against thresholds.** `covered=yes` is under-tagged; void-deck
   routes and mall-interior links are inconsistently mapped. Our covered-network total is therefore an
   *undercount* — which strengthens, not weakens, the invisibility argument, but means absolute
   kilometre figures should be read as lower bounds. (This mirrors, in the open-data domain, exactly the
   coverage-and-bias problem UAL documents for SVI.)
2. **The visibility proxy is geometric, not photographic.** It assumes a road within 20 m implies possible
   capture; real capture depends on Google/Mapillary coverage of that road, occlusion, and camera geometry.
   A follow-up could intersect with actual SVI point coverage via ZenSVI metadata.
3. **Elevation is flattened.** Elevated links and basement concourses near roads are wrongly counted as
   "visible"; the true invisible share is higher. A 3D treatment — thresholds as volumes in a city
   model — is precisely the PhD-scale extension this pilot motivates.

## 10. Next step

Pilot 01 (*Can street-view imagery see a threshold?*) takes three MRT-exit thresholds from the amber
category and the teal category, samples whatever SVI exists via ZenSVI, and checks the imagery against
timed field observation — closing the loop from "invisible in principle" to "what is actually missed."
